# I4 — Full S/B336/L336 ensemble horizontal-flip TTA

Select **T4 GPU**, enable the Colab secret `KAGGLE_API_TOKEN`, and run all cells. The known original F0/I1/I2 Drive folders are prefilled. I3 checkpoints are not used. Original B336/L336 feature caches are required.

This is inference only: average original and horizontally flipped center-crop probabilities per head, then preserve equal S/B/L branch weights. **Validation is diagnostic only; one fixed candidate is exported regardless of its validation score.** The original 0.97959 submission must reproduce before flipped inference starts.

No training or augmented-training-feature extraction occurs. Original B/L predictions reuse saved features. The flipped L336 pass still processes 900 images; it can be slow on T4. Progress includes elapsed/remaining estimates. Completed branch/split probabilities are cached in Drive, so reruns reuse them (an interrupted split restarts). Models load sequentially; L batch defaults to 2. If GPU memory is insufficient, restart the runtime and set LARGE_BATCH_SIZE=1.

Keep the original submission until the candidate is scored. The notebook does not submit to Kaggle.

In [ ]:
%pip -q install scikit-learn pandas pillow kagglehub

In [ ]:
import os
os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')
STORAGE_ROOT=Path('/content/drive/MyDrive/Car-Classification-Benchmark')
F0_RUN_DIR=STORAGE_ROOT/'student_outputs_F0_Ensemble/20261108T091250Z_4ed04fc8'
I1_RUN_DIR=STORAGE_ROOT/'student_outputs_I1_ViTB336/1402790cb5a0'
I2_RUN_DIR=STORAGE_ROOT/'student_outputs_I2_ViTL336/6dcb19f3d7e4'
for folder in (F0_RUN_DIR,I1_RUN_DIR,I2_RUN_DIR):
    assert folder.is_dir(), f'Saved source folder missing: {folder}'
DOWNLOAD_FROM_KAGGLE=True
DATA_ROOT=Path('/content/car_data')
if DOWNLOAD_FROM_KAGGLE:
    import kagglehub
    if not os.environ.get('KAGGLE_API_TOKEN'):
        from google.colab import userdata
        os.environ['KAGGLE_API_TOKEN']=userdata.get('KAGGLE_API_TOKEN')
    DATA_ROOT=Path(kagglehub.competition_download('idnn-swe-3052-assignment-3-flowers-classification'))
for split in ('train','val','test'):
    assert (DATA_ROOT/split).is_dir(), f'Missing dataset split: {DATA_ROOT/split}'
print('Data:',DATA_ROOT)

SMALL_BATCH_SIZE=8
LARGE_BATCH_SIZE=2


In [ ]:
"""I4: fixed original/flip probability TTA on the original equal S/B/L ensemble."""
from pathlib import Path
import gc, hashlib, json, random, subprocess, time, uuid
from urllib.error import HTTPError
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset
from torchvision import datasets, transforms
from torchvision.transforms import InterpolationMode
from PIL import Image



class FlippedTransform:
    def __init__(self,evaluation): self.evaluation=evaluation
    def __call__(self,image): return self.evaluation(image).flip(-1)

def valid_probabilities(value,heads,n):
    assert value.shape==(heads,n,20) and value.dtype==torch.float32
    assert torch.isfinite(value).all() and (value>=0).all() and (value<=1).all()
    assert torch.allclose(value.sum(-1),torch.ones(heads,n),atol=1e-5)

@torch.inference_mode()
def flipped_probabilities(backbone,checkpoints,paths,evaluation,device,batch_size,cache_root,metadata):
    """Resumable branch/split cache; no fitting or validation-based selection."""
    metadata={**metadata,'version':'I4-horizontal-flip-v1','transform':repr(evaluation),
              'torch':str(torch.__version__),'batch_size':batch_size,'device':device}
    signature=hashlib.sha256(json.dumps(metadata,sort_keys=True).encode()).hexdigest()
    path=Path(cache_root)/(signature+'.pt');path.parent.mkdir(parents=True,exist_ok=True)
    if path.exists():
        cached=torch.load(path,map_location='cpu',weights_only=True)
        assert cached['metadata']==metadata
        valid_probabilities(cached['probabilities'],len(checkpoints),len(paths))
        print('I4_FLIP_CACHE_HIT',metadata['branch'],metadata['split'],flush=True)
        return cached['probabilities']
    backbone.eval();assert not any(p.requires_grad for p in backbone.parameters())
    heads=[]
    for c in checkpoints:
        head=build_head(len(c['feature_mean'])).to(device)
        head.load_state_dict(c['head_state_dict']);head.eval()
        heads.append((head,c['feature_mean'].to(device),c['feature_std'].to(device)))
    result=[[] for _ in heads]
    loader=DataLoader(Images(paths,FlippedTransform(evaluation)),batch_size=batch_size,shuffle=False,num_workers=0)
    started=time.perf_counter()
    for index,images in enumerate(loader):
        z=backbone(images.to(device)).float()
        for j,(head,mean,std) in enumerate(heads):
            assert z.shape==(len(images),len(mean))
            result[j].append(head((z-mean)/std).softmax(1).cpu())
        if index%10==0 or index+1==len(loader):
            elapsed=time.perf_counter()-started
            print('I4_FLIP',metadata['branch'],metadata['split'],index+1,'/',len(loader),
                'elapsed_min',round(elapsed/60,1),'remaining_min',round(elapsed/(index+1)*(len(loader)-index-1)/60,1),flush=True)
    result=torch.stack([torch.cat(parts) for parts in result])
    valid_probabilities(result,len(checkpoints),len(paths))
    temporary=path.with_suffix('.'+uuid.uuid4().hex+'.tmp')
    torch.save({'metadata':metadata,'probabilities':result},temporary);temporary.replace(path)
    return result

def run_i4(data_root,f0_run,i1_run,i2_run,storage_root,small_batch_size=8,large_batch_size=2):
    assert torch.cuda.is_available(), 'Select a GPU runtime in Colab before running I4.'
    assert small_batch_size>=1 and large_batch_size>=1
    device='cuda';set_seed(2026)
    mapping,val_paths,target,test_paths,ids=load_data(data_root)
    f0_run,i1_run,i2_run=map(Path,(f0_run,i1_run,i2_run))
    f0=json.loads((f0_run/'ensemble_summary.json').read_text())
    ref=json.loads((i2_run/'summary.json').read_text())
    assert f0['head_seeds']==SEEDS and f0['hub_ref']==HUB_REF and f0['augmentation_seed']==123
    assert np.allclose(f0['weights'],[1/3]*3)
    assert np.allclose(ref['effective_head_weights'],[1/9,1/9,1/9,1/3,1/3])
    expected={Path(p).name:h for p,h in ref['source_checkpoint_sha256'].items()}
    rows={r['seed']:r for r in f0['individual_validation']}
    small_paths=[f0_run/Path(rows[s]['checkpoint']).name for s in SEEDS]
    base_path=i1_run/'I1_dinov2_vitb14_336_p3_e4_aug123_head2026.pt'
    large_path=i2_run/Path(ref['new_checkpoint']).name
    expected[large_path.name]=ref['new_checkpoint_sha256']
    paths=small_paths+[base_path,large_path]
    original={str(p):sha(p) for p in paths}
    for p in paths: assert original[str(p)]==expected[p.name], 'Source checkpoint changed: '+str(p)
    small=[checked_checkpoint(p,'dinov2_vits14',384,s,mapping) for p,s in zip(small_paths,SEEDS)]
    base=checked_checkpoint(base_path,'dinov2_vitb14',768,2026,mapping)
    large=torch.load(large_path,map_location='cpu',weights_only=True)
    assert large['backbone']=='dinov2_vitl14' and large['resolution']==336 and large['class_to_idx']==mapping
    assert base['resolution']==336 and base['feature_metadata']['image_size']==336
    assert large['hub_ref']==HUB_REF and large['feature_metadata']['manifest']==base['feature_metadata']['manifest']
    assert large['feature_mean'].shape==large['feature_std'].shape==(1024,)
    assert torch.isfinite(large['feature_mean']).all() and torch.isfinite(large['feature_std']).all() and (large['feature_std']>0).all()
    validate_source_data(base,data_root,mapping,ids)
    vb,tb=cached_probabilities(base,storage_root,target,ids)
    vl,tl=cached_probabilities(large,storage_root,target,ids)
    reproduce(metrics(target,vb),base['metrics'],'B336 cache')
    reproduce(metrics(target,vl),large['metrics'],'L336 cache')
    evaluation,augmentation=transforms336()
    eval224=transforms.Compose([transforms.Resize(256,interpolation=InterpolationMode.BICUBIC),transforms.CenterCrop(224),transforms.ToTensor(),transforms.Normalize([.485,.456,.406],[.229,.224,.225])])
    encoder=load_official('dinov2_vits14',f0['weights_sha256'],384,device)
    vs=infer(encoder,small,val_paths,eval224,device,small_batch_size)
    ts=infer(encoder,small,test_paths,eval224,device,small_batch_size)
    for i,seed in enumerate(SEEDS): reproduce(metrics(target,vs[i]),rows[seed],f'F0 seed{seed}')
    reproduce(metrics(target,vs.mean(0)),f0['ensemble_validation'],'F0 ensemble')
    current_val=equal_blend(vs.mean(0),vb,vl);current_test=equal_blend(ts.mean(0),tb,tl)
    reproduce(metrics(target,current_val),ref['validation']['new_equal_S_B_L_blend'],'Current 0.97959 ensemble')
    control_path=i2_run/'F0_B336_L336_equal_backbone_submission.csv'
    control=pd.read_csv(control_path,dtype={'image_id':str})
    assert control.image_id.tolist()==ids and np.array_equal(control.label.to_numpy(),current_test.argmax(1).numpy()), 'Winning submission did not reproduce'
    root=Path(storage_root)/'student_outputs_I4_Ensemble_FlipTTA'
    output=root/uuid.uuid4().hex[:12];output.mkdir(parents=True,exist_ok=False)
    print('OUTPUT',output,'DEVICE',torch.cuda.get_device_name(0),flush=True)
    print('Original 0.97959 predictions reproduced. Beginning fixed flip inference.',flush=True)
    flipped={}
    for branch,name,dim,checkpoints,transform,batch_size,digest in [
        ('S224','dinov2_vits14',384,small,eval224,small_batch_size,f0['weights_sha256']),
        ('B336','dinov2_vitb14',768,[base],evaluation,small_batch_size,base['feature_metadata']['weights_sha256']),
        ('L336','dinov2_vitl14',1024,[large],evaluation,large_batch_size,large['feature_metadata']['weights_sha256'])]:
        if branch!='S224':encoder=load_official(name,digest,dim,device)
        pair=[]
        metadata={'branch':branch,'source_checkpoint_sha256':original,
            'hub_ref':HUB_REF,'weights_sha256':digest,'manifest':base['feature_metadata']['manifest'],
            'class_to_idx':mapping,'test_ids':ids}
        for split,images in [('validation',val_paths),('test',test_paths)]:
            pair.append(flipped_probabilities(encoder,checkpoints,images,transform,device,batch_size,root/'flip_cache',{**metadata,'split':split}))
        flipped[branch]=pair
        del encoder;gc.collect();torch.cuda.empty_cache()
    flip_val=equal_blend(flipped['S224'][0].mean(0),flipped['B336'][0][0],flipped['L336'][0][0])
    flip_test=equal_blend(flipped['S224'][1].mean(0),flipped['B336'][1][0],flipped['L336'][1][0])
    candidate_val=.5*(current_val+flip_val);candidate_test=.5*(current_test+flip_test)
    for p in paths:assert sha(p)==original[str(p)],'Source checkpoint modified'
    export(output/'CurrentBest_097959_control.csv',ids,current_test)
    export(output/'I4_S_B336_L336_flip_TTA_submission.csv',ids,candidate_test)
    first=current_val.argmax(1).numpy();second=candidate_val.argmax(1).numpy()
    changed_test=current_test.argmax(1)!=candidate_test.argmax(1)
    report={'experiment':'I4 equal S/B/L original-horizontal-flip probability TTA',
        'rule':'mean original/flip probabilities per head; equal S/B/L branches; F0 heads equal',
        'validation_original':metrics(target,current_val),'validation_TTA':metrics(target,candidate_val),
        'validation_corrected':int(((first!=target)&(second==target)).sum()),
        'validation_regressed':int(((first==target)&(second!=target)).sum()),
        'validation_changed':int((first!=second).sum()),'test_changed':int(changed_test.sum()),
        'source_predictions_reproduced':True,'source_checkpoint_sha256':original,
        'reference_submission_sha256':sha(control_path),'historical_best_kaggle':.97959,
        'export_rule':'always export after integrity checks; validation diagnostic only',
        'small_batch_size':small_batch_size,'large_batch_size':large_batch_size,
        'test_labels_used':False,'training':False,'submitted':False}
    pd.DataFrame({'image_id':ids,'original_label':current_test.argmax(1).numpy(),
        'tta_label':candidate_test.argmax(1).numpy(),'changed':changed_test.numpy()}).to_csv(output/'test_prediction_changes.csv',index=False)
    np.savez_compressed(output/'probabilities.npz',validation_original=current_val.numpy(),validation_tta=candidate_val.numpy(),
        test_original=current_test.numpy(),test_tta=candidate_test.numpy(),targets=target)
    (output/'summary.json').write_text(json.dumps(report,indent=2))
    print('COMPLETE',output,json.dumps(report),flush=True)
    return output,report

HUB_REF='7764ea0f912e53c92e82eb78a2a1631e92725fc8'
SEEDS=[42,123,2026]
def sha(path):
    h=hashlib.sha256()
    with open(path,'rb') as stream:
        for chunk in iter(lambda:stream.read(1024*1024),b''):h.update(chunk)
    return h.hexdigest()

def weight_digest(model):
    h=hashlib.sha256()
    for name,value in sorted(model.state_dict().items()):
        h.update(name.encode());h.update(value.detach().cpu().contiguous().numpy().tobytes())
    return h.hexdigest()

def build_head(dim):
    return nn.Sequential(nn.Linear(dim,128),nn.BatchNorm1d(128),nn.ReLU(),nn.Dropout(.3),nn.Linear(128,20))

class Images(torch.utils.data.Dataset):
    def __init__(self,paths,transform):self.paths,self.transform=paths,transform
    def __len__(self):return len(self.paths)
    def __getitem__(self,index):
        with Image.open(self.paths[index]) as image:return self.transform(image.convert('RGB'))

def load_data(root):
    root=Path(root)
    train,val=[datasets.ImageFolder(root/name) for name in ('train','val')]
    assert len(train)==700 and len(val)==200 and len(train.classes)==20
    assert train.class_to_idx==val.class_to_idx
    assert all(name.startswith(f'{i:03d}_') for i,name in enumerate(train.classes))
    sample=pd.read_csv(root/'sample_submission.csv',dtype={'image_id':str})
    assert {'image_id','label'}<=set(sample.columns) and sample.image_id.is_unique and sample.image_id.notna().all()
    files=sorted(p for p in (root/'test').iterdir() if p.is_file() and p.suffix.lower() in {'.jpg','.jpeg','.png','.bmp','.webp'})
    by_id={p.stem:p for p in files};ids=sample.image_id.tolist()
    assert len(by_id)==len(files) and set(by_id)==set(ids)
    return train.class_to_idx,[Path(p) for p,_ in val.samples],np.asarray(val.targets),[by_id[i] for i in ids],ids

def checked_checkpoint(path,model,dim,seed,mapping):
    p=torch.load(path,map_location='cpu',weights_only=True)
    assert p['backbone']==model and p['seed_aug']==123 and p['seed_head']==seed
    assert p['class_to_idx']==mapping
    for key in ('feature_mean','feature_std'):
        assert p[key].shape==(dim,) and torch.isfinite(p[key]).all()
    assert (p['feature_std']>0).all()
    expected={'hidden_dim':128,'use_bn':True,'dropout':.3}
    if dim==768:expected={'input_dim':768,**expected,'num_classes':20}
    assert p['head_config']==expected,'Checkpoint is not the requested E4 architecture'
    build_head(dim).load_state_dict(p['head_state_dict'],strict=True)
    return p

@torch.inference_mode()
def infer(backbone,checkpoints,paths,transform,device,batch_size=8):
    backbone.eval();assert not any(p.requires_grad for p in backbone.parameters())
    heads=[]
    for c in checkpoints:
        h=build_head(len(c['feature_mean'])).to(device)
        h.load_state_dict(c['head_state_dict'],strict=True);h.eval()
        heads.append((h,c['feature_mean'].to(device),c['feature_std'].to(device)))
    results=[[] for _ in heads]
    loader=DataLoader(Images(paths,transform),batch_size=batch_size,shuffle=False,num_workers=0)
    for index,images in enumerate(loader):
        features=backbone(images.to(device)).float()
        for j,(head,mean,std) in enumerate(heads):
            assert features.shape==(len(images),len(mean))
            p=head((features-mean)/std).softmax(1)
            assert p.shape==(len(images),20) and torch.isfinite(p).all()
            results[j].append(p.cpu())
        if index%40==0:print('PREDICT_BATCH',index+1,'/',len(loader),flush=True)
    return torch.stack([torch.cat(parts) for parts in results])

def metrics(target,probabilities):
    from sklearn.metrics import accuracy_score,f1_score
    pred=probabilities.argmax(1).numpy()
    return {'accuracy':float(accuracy_score(target,pred)),
        'macro_f1':float(f1_score(target,pred,labels=list(range(20)),average='macro',zero_division=0)),
        'correct':int((pred==target).sum()),'count':len(target)}

def reproduce(measured,expected,name):
    assert measured['correct']==expected['correct'] and abs(measured['accuracy']-expected['accuracy'])<1e-12, (
        f'{name} starting accuracy did not reproduce; diagnose checkpoint/source/preprocessing before exporting')

def export(path,ids,probabilities):
    pred=probabilities.argmax(1).numpy().astype(int)
    assert len(pred)==len(ids)==len(set(ids)) and ((pred>=0)&(pred<20)).all()
    pd.DataFrame({'image_id':list(ids),'label':pred}).to_csv(path,index=False,mode='x')

def set_seed(seed):
    random.seed(seed);np.random.seed(seed);torch.manual_seed(seed)
    if torch.cuda.is_available():torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark=False;torch.backends.cudnn.deterministic=True
    torch.use_deterministic_algorithms(True)

def transforms336():
    tail=[transforms.ToTensor(),transforms.Normalize([.485,.456,.406],[.229,.224,.225])]
    evaluation=transforms.Compose([transforms.Resize(384,interpolation=InterpolationMode.BICUBIC),transforms.CenterCrop(336),*tail])
    training=transforms.Compose([transforms.RandomResizedCrop(336,scale=(.8,1.),ratio=(.9,1.1),interpolation=InterpolationMode.BICUBIC),transforms.RandomHorizontalFlip(.5),*tail])
    return evaluation,training

def validate_source_data(base,data_root,mapping,ids):
    meta=base['feature_metadata']
    assert meta.get('backbone',meta.get('model'))=='dinov2_vitb14'
    assert base['hub_ref']==HUB_REF and meta['class_to_idx']==mapping and meta['test_ids']==ids
    assert meta['feature_dim']==768 and meta['seed_aug']==123 and meta['n_aug']==4 and meta['include_original'] is True
    root=Path(data_root).resolve()
    for split in meta['manifest']:
        for entry in (split['files'] if isinstance(split,dict) else split):
            path=(root/entry['path']).resolve()
            assert path.is_relative_to(root) and path.is_file() and sha(path)==entry['sha256'],'Data differs from original I0 run'

def load_official(name,expected,dim,device):
    try:
        model=torch.hub.load(f'facebookresearch/dinov2:{HUB_REF}',name,pretrained=True,trust_repo=True,skip_validation=True)
    except HTTPError as exc:
        if exc.code not in (502,503,504):raise
        root=Path(torch.hub.get_dir())/('dinov2_git_'+HUB_REF[:12])
        if not root.exists():
            subprocess.run(['git','clone','--filter=blob:none','--no-checkout','https://github.com/facebookresearch/dinov2.git',str(root)],check=True)
        subprocess.run(['git','-C',str(root),'checkout','--detach',HUB_REF],check=True)
        assert subprocess.check_output(['git','-C',str(root),'rev-parse','HEAD'],text=True).strip()==HUB_REF
        model=torch.hub.load(str(root),name,source='local',pretrained=True)
    model.requires_grad_(False).eval()
    assert model.embed_dim==dim
    if expected is not None:assert weight_digest(model)==expected,'Backbone does not match original source weights'
    return model.to(device)

@torch.inference_mode()
def cached_probabilities(checkpoint,storage_root,target,ids):
    signature=checkpoint['feature_signature']
    assert signature==hashlib.sha256(json.dumps(checkpoint['feature_metadata'],sort_keys=True).encode()).hexdigest()
    matches=list(Path(storage_root).glob('student_outputs_I*/cache_336/*_'+signature+'.pt'))
    assert len(matches)==1, 'Required original feature cache missing/ambiguous: '+signature+'; restore it from the original run, do not re-extract L336.'
    cached=torch.load(matches[0],map_location='cpu',weights_only=True)
    assert cached['metadata']==checkpoint['feature_metadata']
    assert cached['metadata']['test_ids']==ids and cached['metadata']['image_size']==336
    assert np.array_equal(cached['y_val'].numpy(),target)
    head=build_head(len(checkpoint['feature_mean']));head.load_state_dict(checkpoint['head_state_dict']);head.eval()
    result=[]
    for key,n in [('X_val',len(target)),('X_test',len(ids))]:
        x=cached[key]
        assert x.shape==(n,len(checkpoint['feature_mean'])) and x.dtype==torch.float32 and torch.isfinite(x).all()
        result.append(head((x-checkpoint['feature_mean'])/checkpoint['feature_std']).softmax(1))
    return result

def equal_blend(small,base,large):
    assert small.shape==base.shape==large.shape and small.ndim==2 and small.shape[1]==20
    for p in (small,base,large):
        assert torch.isfinite(p).all() and (p>=0).all() and (p<=1).all()
        assert torch.allclose(p.sum(1),torch.ones(len(p)),atol=1e-5)
    return (small+base+large)/3

In [ ]:
output,summary=run_i4(DATA_ROOT,F0_RUN_DIR,I1_RUN_DIR,I2_RUN_DIR,STORAGE_ROOT,SMALL_BATCH_SIZE,LARGE_BATCH_SIZE)
from IPython.display import display
display(pd.DataFrame([{'inference':name,**summary[key]} for name,key in [('original','validation_original'),('flip TTA','validation_TTA')]]))
print('Changed test predictions:',summary['test_changed'])
print('New candidate (not submitted):',output/'I4_S_B336_L336_flip_TTA_submission.csv')
print('Original control:',output/'CurrentBest_097959_control.csv')
print('Keep the full output folder:',output)